In [ ]:
%run "./02_EMS_preprocessing.ipynb"

import numpy as np
import pandas as pd

files = {
    "train": PROCESSED_DIR / "train_with_baselines.csv",
    "validation": PROCESSED_DIR / "validation_with_baselines.csv",
    "test": PROCESSED_DIR / "test_with_baselines.csv",
}

for path in files.values():
    if not path.exists():
        raise FileNotFoundError(f"Fichier absent : {path}")

datasets = {
    name: pd.read_csv(path)
    for name, path in files.items()
}

for name, data in datasets.items():
    print(name, data.shape)

ROOT_DIR  : C:\Users\Admin\Desktop\Projet_Artemis2
DATA_FILE : C:\Users\Admin\Desktop\Projet_Artemis2\data\Artemis.csv | existe: True
RANDOM_SEED: 42 | DEVICE: cuda
CONFIGURATION DU PROJET HESS

BATTERIE ÉNERGIE
Énergie          : 13709.89 Wh
Tension          : 450.00 V
Capacité         : 30.4664 Ah
Masse            : 55.12 kg
Courant recharge : -14.00 A
Courant décharge : 28.00 A
Configuration    : 125S7P

BATTERIE PUISSANCE
Énergie          : 2987.12 Wh
Tension          : 402.60 V
Capacité         : 7.4196 Ah
Masse            : 50.02 kg
Courant recharge : -130.00 A
Courant décharge : 400.00 A
Configuration    : 122S2P

HESS
Énergie totale   : 16697.01 Wh
Masse totale     : 105.14 kg
Tension du bus   : 402.60 V
Puissance min    : -58638.00 W
Puissance max    : 173640.00 W

MODÈLES
LSTM seul        : 7 → 64 → 3
LSTM NS          : 15 → 64 → 3
GNN simple       : 12 → 32 → 1
MLP simple       : 5 → 64 → 32 → 1
MLP NS           : 17 → 64 → 32 → 1

Device           : cuda
Fichier          : 

Vérification des données

In [2]:
required_cols = [
    TIME_COL,
    "SOC_EB",
    "SOC_PB",
    "hasPower",
    "P_conv",
]

for name, data in datasets.items():
    missing = sorted(
        set(required_cols) - set(data.columns)
    )

    if missing:
        raise ValueError(
            f"{name} : colonnes absentes {missing}"
        )

    values = data[required_cols].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            f"{name} contient des valeurs non finies."
        )

print("Données validées.")

Données validées.


Génération des états symboliques

In [3]:
def add_symbolic_states(data):
    result = data.copy()

    p_dem = result["hasPower"]
    p_conv = result["P_conv"]

    result["EB_available"] = (
        result["SOC_EB"] > SOC_EB_MIN
    ).astype(np.int8)

    result["PB_available"] = (
        result["SOC_PB"] > SOC_PB_MIN
    ).astype(np.int8)

    result["EB_low_SOC"] = (
        result["SOC_EB"] <= SOC_LOW_THRESHOLD
    ).astype(np.int8)

    result["PB_low_SOC"] = (
        result["SOC_PB"] <= SOC_LOW_THRESHOLD
    ).astype(np.int8)

    result["high_power_demand"] = (
        p_dem.abs() >= HIGH_POWER_THRESHOLD_W
    ).astype(np.int8)

    result["regenerative_braking"] = (
        p_dem < -EPS_POWER_W
    ).astype(np.int8)

    result["zero_power_demand"] = (
        p_dem.abs() <= EPS_POWER_W
    ).astype(np.int8)

    utilization = np.where(
        p_conv >= 0,
        p_conv / P_CONV_MAX_W,
        p_conv.abs() / abs(P_CONV_MIN_W),
    )

    result["converter_utilization"] = utilization

    result["converter_risk"] = (
        utilization >= CONVERTER_RISK_THRESHOLD
    ).astype(np.int8)

    return result

Application par split

In [4]:

EPS_POWER_W = 100.0
SOC_LOW_THRESHOLD = 0.30
HIGH_POWER_THRESHOLD_W = 20000.0
CONVERTER_RISK_THRESHOLD = 0.85

assert 0.0 < CONVERTER_RISK_THRESHOLD < 1.0



In [5]:
symbolic_datasets = {
    name: add_symbolic_states(data)
    for name, data in datasets.items()
}

for name, data in symbolic_datasets.items():
    print(name, data.shape)

train (7265, 39)
validation (3632, 39)
test (3633, 39)


Proportions par split

In [6]:
summary_rows = []

for split_name, data in symbolic_datasets.items():
    for indicator in SYMBOLIC_COLS:
        summary_rows.append({
            "split": split_name,
            "indicator": indicator,
            "active_count": int(data[indicator].sum()),
            "active_percent": float(
                100.0 * data[indicator].mean()
            ),
        })

summary_df = pd.DataFrame(summary_rows)

display(
    summary_df.pivot(
        index="indicator",
        columns="split",
        values="active_percent",
    ).round(2)
)

split,test,train,validation
indicator,,,
EB_available,87.92,100.00,100.00
EB_low_SOC,67.08,0.00,0.00
PB_available,100.00,100.00,100.00
PB_low_SOC,5.67,0.00,0.00
converter_risk,22.38,24.58,27.18
high_power_demand,8.78,7.39,9.00
regenerative_braking,28.19,28.89,28.50
zero_power_demand,15.61,18.46,14.29


Proportions globales

In [7]:
all_symbolic = pd.concat(
    [
        data.assign(split=name)
        for name, data in symbolic_datasets.items()
    ],
    ignore_index=True,
)

global_summary = pd.DataFrame({
    "indicator": SYMBOLIC_COLS,
    "active_count": [
        int(all_symbolic[col].sum())
        for col in SYMBOLIC_COLS
    ],
    "active_percent": [
        float(100.0 * all_symbolic[col].mean())
        for col in SYMBOLIC_COLS
    ],
})

display(global_summary.round(2))

,indicator,active_count,active_percent
0,EB_available,14091,96.98
1,PB_available,14530,100.00
2,EB_low_SOC,2437,16.77
3,PB_low_SOC,206,1.42
4,high_power_demand,1183,8.14
5,regenerative_braking,4158,28.62
6,zero_power_demand,2427,16.70
7,converter_risk,3586,24.68


 Vérifications logiques

In [8]:

for name, data in symbolic_datasets.items():
    for col in SYMBOLIC_COLS:
        values = set(data[col].unique())

        if not values.issubset({0, 1}):
            raise ValueError(
                f"{name} : {col} n'est pas binaire."
            )

    conflict_zero_regen = (
        data["zero_power_demand"].eq(1)
        & data["regenerative_braking"].eq(1)
    )

    conflict_zero_high = (
        data["zero_power_demand"].eq(1)
        & data["high_power_demand"].eq(1)
    )

    if conflict_zero_regen.any():
        raise ValueError(
            f"{name} : conflit zero/regen."
        )

    if conflict_zero_high.any():
        raise ValueError(
            f"{name} : conflit zero/high power."
        )

print("Cohérence logique validée.")

Cohérence logique validée.


Détection des indicateurs constants

In [9]:
train_symbolic = symbolic_datasets["train"]

constant_train_cols = [
    col
    for col in SYMBOLIC_COLS
    if train_symbolic[col].nunique() <= 1
]

print(
    "Indicateurs constants dans le train :",
    constant_train_cols,
)

Indicateurs constants dans le train : ['EB_available', 'PB_available', 'EB_low_SOC', 'PB_low_SOC']


Fonction d’affichage d’un scénario

In [10]:
scenario_cols = [
    "split",
    TIME_COL,
    "SOC_EB",
    "SOC_PB",
    "hasPower",
    "P_conv",
    "converter_utilization",
] + SYMBOLIC_COLS


def show_case(data, index, label):
    print(f"\n{label}")
    display(
        data.loc[
            [index],
            scenario_cols,
        ].T
    )

Vérification des scénarios

In [11]:
idx_pb_low = (
    all_symbolic["SOC_PB"] - SOC_PB_MIN
).abs().idxmin()

idx_max_power = all_symbolic[
    "hasPower"
].idxmax()

idx_zero_power = all_symbolic[
    "hasPower"
].abs().idxmin()

idx_max_regen = all_symbolic[
    "hasPower"
].idxmin()

show_case(
    all_symbolic,
    idx_pb_low,
    "SOC PB proche de la limite basse",
)

show_case(
    all_symbolic,
    idx_max_power,
    "Forte traction",
)

show_case(
    all_symbolic,
    idx_zero_power,
    "Demande quasi nulle",
)

show_case(
    all_symbolic,
    idx_max_regen,
    "Freinage régénératif maximal",
)


SOC PB proche de la limite basse


,14497
split,test
time,14498
SOC_EB,0.2
SOC_PB,0.202598
hasPower,-517.22
P_conv,-54.4805
converter_utilization,0.071685
EB_available,0
PB_available,1
EB_low_SOC,1



Forte traction


,1248
split,train
time,1249
SOC_EB,0.959395
SOC_PB,0.984669
hasPower,44230.33
P_conv,1327.2
converter_utilization,0.873158
EB_available,1
PB_available,1
EB_low_SOC,0



Demande quasi nulle


,0
split,train
time,1
SOC_EB,1.0
SOC_PB,1.0
hasPower,0.0
P_conv,0.0
converter_utilization,0.0
EB_available,1
PB_available,1
EB_low_SOC,0



Freinage régénératif maximal


,1283
split,train
time,1284
SOC_EB,0.956916
SOC_PB,0.974919
hasPower,-46000.0
P_conv,-663.6
converter_utilization,0.873158
EB_available,1
PB_available,1
EB_low_SOC,0


Deux SOC faibles

In [12]:
two_low_soc = (
    (all_symbolic["SOC_EB"] <= 0.25)
    & (all_symbolic["SOC_PB"] <= 0.30)
)

print(
    "Deux SOC faibles :",
    int(two_low_soc.sum()),
    "lignes",
)

if two_low_soc.any():
    first_index = all_symbolic.index[
        two_low_soc
    ][0]

    show_case(
        all_symbolic,
        first_index,
        "Exemple avec deux SOC faibles",
    )

Deux SOC faibles : 206 lignes

Exemple avec deux SOC faibles


,14324
split,test
time,14325
SOC_EB,0.2
SOC_PB,0.299871
hasPower,17441.61
P_conv,0.0
converter_utilization,0.0
EB_available,0
PB_available,1
EB_low_SOC,1


Distribution du convertisseur

In [13]:
print(
    "Utilisation maximale :",
    all_symbolic[
        "converter_utilization"
    ].max(),
)

for threshold in [0.5, 0.75, 0.80, 0.85, 0.90]:
    active_percent = 100.0 * (
        all_symbolic[
            "converter_utilization"
        ] >= threshold
    ).mean()

    print(
        f"Seuil {threshold:.2f} : "
        f"{active_percent:.2f} % actifs"
    )

Utilisation maximale : 0.8731578947368421
Seuil 0.50 : 42.44 % actifs
Seuil 0.75 : 27.61 % actifs
Seuil 0.80 : 25.93 % actifs
Seuil 0.85 : 24.68 % actifs
Seuil 0.90 : 0.00 % actifs


 Sauvegarde

In [14]:

output_files = {
    "train": (
        PROCESSED_DIR
        / "train_with_symbolic_states.csv"
    ),
    "validation": (
        PROCESSED_DIR
        / "validation_with_symbolic_states.csv"
    ),
    "test": (
        PROCESSED_DIR
        / "test_with_symbolic_states.csv"
    ),
}

for name, path in output_files.items():
    symbolic_datasets[name].to_csv(
        path,
        index=False,
    )

    print(name, "->", path)

summary_df.to_csv(
    TABLES_DIR
    / "symbolic_states_by_split.csv",
    index=False,
)

global_summary.to_csv(
    TABLES_DIR
    / "symbolic_states_global.csv",
    index=False,
)

print("États symboliques enregistrés.")



train -> C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\train_with_symbolic_states.csv
validation -> C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\validation_with_symbolic_states.csv
test -> C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\test_with_symbolic_states.csv
États symboliques enregistrés.
